# Dijkstra vs A* — Benchmark Comparison

This notebook compares Dijkstra's algorithm and A* on the delivery route graph, measuring nodes expanded, runtime, and path cost across multiple start/end pairs.

In [ ]:
import sys
sys.path.append("..")  # so src/ imports work from notebooks/

from src.graph.builder import build_from_json
from src.utils.benchmark import compare
import pandas as pd
import matplotlib.pyplot as plt

graph = build_from_json("../data/sample_graph.json")

## Run comparisons across multiple start/end pairs

In [ ]:
test_pairs = [
    ("N1", "N20"), ("N1", "N16"), ("N6", "N15"),
    ("N13", "N9"), ("N7", "N10"), ("N18", "N5"),
]

rows = []
for start, end in test_pairs:
    result = compare(graph, start, end, mode="distance")
    for algo_name, data in result.items():
        rows.append({
            "pair": f"{start}->{end}",
            "algorithm": algo_name,
            "cost": data["cost"],
            "nodes_expanded": data["nodes_expanded"],
            "time_seconds": data["time_seconds"],
        })

df = pd.DataFrame(rows)
df

## Chart: Nodes expanded comparison

This is the key chart — it shows whether A*'s heuristic actually reduces search space compared to Dijkstra's uniform exploration.

In [ ]:
pivot = df.pivot(index="pair", columns="algorithm", values="nodes_expanded")
pivot.plot(kind="bar", figsize=(10, 5), title="Nodes Expanded: Dijkstra vs A*")
plt.ylabel("Nodes Expanded")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../data/nodes_expanded_comparison.png")
plt.show()

## Chart: Runtime comparison

In [ ]:
pivot_time = df.pivot(index="pair", columns="algorithm", values="time_seconds")
pivot_time.plot(kind="bar", figsize=(10, 5), title="Runtime: Dijkstra vs A*")
plt.ylabel("Time (seconds)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("../data/runtime_comparison.png")
plt.show()

## Correctness check: both algorithms should agree on optimal cost

Even though A* may expand fewer nodes, it should still find a path with the *same* total cost as Dijkstra (since A* with an admissible heuristic is still optimal). Any nonzero difference here signals a bug or a non-admissible heuristic.

In [ ]:
cost_check = df.pivot(index="pair", columns="algorithm", values="cost")
cost_check["difference"] = (cost_check["dijkstra"] - cost_check["astar"]).abs()
cost_check

## Summary statistics

In [ ]:
summary = df.groupby("algorithm")[["nodes_expanded", "time_seconds"]].mean()
print("Average performance across all test pairs:")
summary

## Conclusion

- Dijkstra explores uniformly outward from the start node, guaranteeing the shortest path but expanding more nodes than necessary.
- A* uses a haversine-distance heuristic to bias search toward the goal, typically expanding fewer nodes for the same optimal cost.
- The gap between the two grows with graph size and the geographic distance between start/end — on this 20-node graph the difference is modest, but it becomes significant on real city-scale OSM graphs.